# PPRL Pipeline - Data Preparation

## Objective
To load, clean, and standardize EHR data for record linkage.

## Why this matters
Data quality directly affects linkage quality, especially in low-resource settings with inconsistent identifiers.

In [ ]:
# =================== BOOTSTRAP CELL ===================
# Standard setup for all notebooks
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[0]  # assumes notebooks are in a subfolder
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ========================================================
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# from src.config.variables import COVARIATES

# ========================================================
# Optional for warnings and nicer plots
import warnings
warnings.filterwarnings("ignore")
sns.set(style="whitegrid")

import sys
from pathlib import Path

# ========================================================
# 1️⃣ Ensure project root is in Python path
# Adjust this if your notebooks are nested deeper
PROJECT_ROOT = Path.cwd().parents[0]  # assumes notebooks are in a subfolder
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ========================================================
# 2️⃣ Import helper to load paths
from src.utils.helpers import load_paths

# ========================================================
# 3️⃣ Load paths from config.yaml (works regardless of notebook location)
paths = load_paths()

# ========================================================
# 4️⃣ Optionally, print paths to confirm
for key, value in paths.items():
    print(f"{key}: {value}")

# ========================================================
# 5️⃣ Now you can use these paths in your notebook:
# Example:
SYNTHETIC_DATA_DIR = paths['SYNTHETIC_DATA_DIR']
PROCESSED_DATA_DIR = paths['PROCESSED_DATA_DIR']
# FIG_DIR = paths['FIG_DIR']

# ========================================================

### Loading data
Prevents memory crashes
<br> Enables processing very large datasets

In [ ]:
import pandas as pd

# Load synthetic data
df_A = pd.read_csv(SYNTHETIC_DATA_DIR / "dataset_A.csv")
df_B = pd.read_csv(SYNTHETIC_DATA_DIR / "dataset_B.csv")

### Key Observations
- Check for missing values
- Inspect identifier consistency

In [ ]:
def preprocess(df):
    df = df.copy()

    df["first_name"] = df["first_name"].str.lower().str.strip()
    df["last_name"] = df["last_name"].str.lower().str.strip()
    df["dob"] = df["dob"].astype(str).fillna("")

    df["combined"] = df["first_name"] + df["last_name"] + df["dob"]

    return df

df_A = preprocess(df_A)
df_B = preprocess(df_B)

# Save processed data
df_A.to_pickle(PROCESSED_DATA_DIR / "df_A_preprocessed.pkl")
df_B.to_pickle(PROCESSED_DATA_DIR / "df_B_preprocessed.pkl")

### Combine identifiers

Creating a single linkage key which simplifies encoding and allows bloom filters to capture joint information

### Insight
Combining identifiers allows joint encoding and improves linkage robustness.